# Basketball Path-B training (real labeled data only)
Downloads the authenticated Roboflow Universe dataset, validates its real class map, fine-tunes YOLO11n, and copies actual artifacts to Drive. No placeholder data or metrics.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "GPU required: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
import os
REPO_URL = "https://github.com/wheredawoodat949/AI-Hackathon"
BRANCH = "basketball"
if not os.path.isdir("/content/AI-Hackathon"):
    !git clone --branch {BRANCH} {REPO_URL}.git /content/AI-Hackathon
%cd /content/AI-Hackathon
!git pull --ff-only origin {BRANCH}
!pip install -q -e . --no-deps
!pip install -q ultralytics "roboflow>=1.3,<2" PyYAML

## Authenticate
Recommended: add a private Colab secret named `ROBOFLOW_API_KEY`. The fallback prompt is hidden. Never paste the key into a committed cell.

In [ ]:
import getpass, os
try:
    from google.colab import userdata
    key = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    key = None
if not key:
    key = getpass.getpass("Roboflow API key (hidden): ")
assert key, "ROBOFLOW_API_KEY is required"
os.environ["ROBOFLOW_API_KEY"] = key

Before continuing, open the Universe project and review its current dataset license. The command below queries real authenticated versions and prints the selected version and exact class map.

In [ ]:
!python -m src.training.basketball download
!python -m src.training.basketball inspect --data data/basketball_det/data.yaml

## Train
This can take time. The cell saves only returned Ultralytics artifacts. Adjust epochs/batch only if GPU memory or hackathon timing requires it.

In [ ]:
!python -m src.training.basketball train \
  --data data/basketball_det/data.yaml \
  --base-model yolo11n.pt --epochs 50 --imgsz 640 --batch 16 --device cuda

In [ ]:
import glob, json
summaries = sorted(glob.glob("runs/basketball/**/phase3_summary.json", recursive=True))
assert summaries, "Training summary not found; inspect the training cell output"
summary = json.load(open(summaries[-1]))
print(json.dumps(summary, indent=2))

## Persist real artifacts to your Drive
The repo paths are gitignored. This copies the actual checkpoint and summary before the Colab runtime expires.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
drive.mount("/content/drive")
destination = Path("/content/drive/MyDrive/AI-Hackathon/basketball_phase3")
destination.mkdir(parents=True, exist_ok=True)
shutil.copy2(summary["best_checkpoint"], destination / "basketball_best.pt")
shutil.copy2(summaries[-1], destination / "phase3_summary.json")
print(destination)